## Bronze → silver — what changes between layers

Bronze is *raw and append-only*. Silver is *clean, conformed, deduplicated, and stable*. The transformation between the two layers is what this whole module is about — and on the exam it's the heaviest single topic.

Six things almost every bronze-to-silver step does:

1. **Drop or replace nulls** in required columns — an `INSERT` with a null `customer_id` should never reach silver.
2. **Cast types** — bronze often lands everything as `STRING`; silver pins `DECIMAL`, `TIMESTAMP`, `BOOLEAN`.
3. **Normalise values** — uppercase categories, trim whitespace, standardise country codes.
4. **Deduplicate** — the same transaction can arrive twice if upstream retried; keep one row per natural key.
5. **Flatten nested structs** that bronze landed verbatim from JSON.
6. **Join in conformed dimensions** — the silver row carries the customer's city directly, not just a foreign key.

What silver does **not** do: pre-aggregate. Sums-by-day and customer-360 rollups belong in gold; silver stays at the *grain of one event per row*, just cleaner.

**Reading bronze** — two equivalent shapes the exam may show either side of:

```python
df = spark.read.table("fintech_dev.bronze.card_transactions_raw")
```

```sql
SELECT * FROM fintech_dev.bronze.card_transactions_raw
```

Both return a **lazy** DataFrame — reading a Delta table touches no bytes until an action runs, and Catalyst pushes column pruning, predicate pushdown, and partition pruning into the read. For *reproducible* silver, pin a version with `.option("versionAsOf", 12)`; for continuously-updating silver, read with `spark.readStream.table(...)`.
